## Dice-Sorensen-Koef

In [ ]:
!pip -q install medpy

  Preparing metadata (setup.py) ... done


In [ ]:
from medpy.metric.binary import dc # Dice Coefficient

Der Dice-Koeffizient (DSC) misst, wie ähnlich zwei Bereiche (oder Segmentierungen) in einem Bild sind. Man kann ihn sich vorstellen als:

$$ DSC = \frac{2|A \cap B|}{|A| + |B|} $$

wobei,
- $A$ die Menge der Pixel im ersten Segmentierungsergebnis
- $|A|$ die Anzahl der Elemente (Pixel) in Menge $A$ ist.


## Färbungstrennung (Hematoxylin und Eosin)

In [ ]:
import cv2
import numpy as np

def deconvolve_stain(image):
    """
    Diese Funktion trennt das Hematoxylin- und Eosin-Färbesignal.
    """
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # RGB-Farbraum
    stain_matrix = np.array([[0.650, 0.072, 0.272],   # H-Channel
                             [0.198, 0.907, 0.132]])  # E-Channel
    stains = np.dot(image_rgb.reshape(-1, 3), stain_matrix.T) # RGB-Transformation
    hematoxylin = stains[:, 0].reshape(image.shape[:2])
    eosin = stains[:, 1].reshape(image.shape[:2])
    return hematoxylin, eosin

In [21]:
%pip install -q pillow

In [33]:
from PIL import Image
import numpy as np
from skimage.transform import resize
import os

def load_and_preprocess_image_tif(image_path, mask_path, target_size=(256, 256)):
    """
    Diese Funktion lädt und skaliert Bilder und Masken im TIFF-Format auf eine einheitliche Größe.
    """
    # Lade das Bild im TIFF-Format
    image = Image.open(image_path)
    mask = Image.open(mask_path)   # Lade die zugehörige Maske
    # Konvertiere die PIL-Bilder in NumPy-Arrays
    image_array = np.array(image)
    mask_array = np.array(mask)
    # Skaliere die Bilder und Masken auf die Zielgröße
    image_resized = resize(image_array, target_size, mode='constant', preserve_range=True)
    mask_resized = resize(mask_array, target_size, mode='constant', preserve_range=True)
    # Normalisiere das Bild
    image_normalized = image_resized / 255.0
    mask_normalized = mask_resized / 255.0
    mask_binary = np.round(mask_normalized)  # Binär-Maske
    return image_normalized, mask_binary

train_images = []
train_masks = []
train_dir = '/content/TrainData'

# Laden der TIFF-Bilder und Masken
for file_name in os.listdir(train_dir):
    if file_name.endswith('.tif') and '_mask' not in file_name:
        image_path = os.path.join(train_dir, file_name)
        mask_path = image_path.replace('.tif', '_mask.tif')
        image, mask = load_and_preprocess_image_tif(image_path, mask_path)
        train_images.append(image)
        train_masks.append(mask)

train_images = np.array(train_images)
train_masks = np.array(train_masks)

In [39]:
print(len(train_images) + len(train_masks) == 74)

True
